# Investigação dos dados e consultas SQL

Este notebook lê o arquivo com os dados integrados e realiza consultas SQL para responder perguntas sobre o conjunto de dados para testar as hipóteses definidas na etapa de 'Preparação e integração dos dados'.

### Consultas

**1. Quantas interrupções ocorreram nos últimos 5 anos? Quais estados da Região Sul tiveram uma proporção maior de interrupções?**

A primeira consulta visa medir o volume de interrupções no período de análise, o que indica se há eventos suficientes para treinar um modelo. Já a segunda compara os estados pela proporção das perturbações em que estiveram envolvidos, para verificar a concentração da parcela das interrupções no Sul do país. Como uma perturbação pode atingir vários estados, essas proporções não somam 100%, e elas mostram onde há mais interrupções, e não onde o risco é maior em relação ao tamanho da rede de cada estado.

In [ ]:
from pathlib import Path

import duckdb
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chisquare

interruption_file = Path("../data/raw/ONS/INTERRUPCAO_CARGA.csv")
processed_file = Path("../data/processed/ons_inmet_sul.parquet")

INICIO = "2021-01-01"
FIM = "2025-12-31"

In [ ]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_parquet('{processed_file}')
    WHERE TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    SELECT SUM(n_perturbacoes) AS perturbacoes
    FROM read_parquet('{processed_file}')
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
""").df()

In [ ]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_csv_auto('{interruption_file}', delim=';', header=true)
    WHERE id_subsistema = 'S'
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    WITH periodo AS ({periodo})
    SELECT estado,
           COUNT(DISTINCT perturbacao) AS perturbacoes,
           ROUND(100.0 * COUNT(DISTINCT perturbacao) / (SELECT COUNT(DISTINCT perturbacao) FROM periodo), 1) AS pct_perturbacoes
    FROM periodo
    GROUP BY estado
    ORDER BY perturbacoes DESC
""").df()

Foram retornados 305 perturbações no Subsistema Sul entre 2021 e 2025, cerca de 61 por ano. Como são 1.826 dias no período, no máximo 17% deles têm alguma perturbação (menos, se houver dias com mais de uma). A maioria dos dias não tem interrupção, então a variável-alvo de alta interrupção será desbalanceada. (Mudar para todas as regiões?)

Por estado, o Rio Grande do Sul esteve envolvido em 68,9% das perturbações, o Paraná em 27,9% e Santa Catarina em 6,2%. A soma passa de 100% porque 210, 85 e 19 totalizam 314, mais que as 305 perturbações, e a diferença vem de perturbações que atingiram mais de um estado. A concentração no RS não prova que o estado tenha mais risco, já que a proporção não considera o tamanho da rede de cada estado. Santa Catarina tem poucos eventos comparado aos outros estados (19), o que torna inviável um modelo por estado. (Isso significa que vamos ter que prever se vai ter interrupção só na região? Visto proporção desbalanceada por estados.)

**2. Quantas interrupções foram acompanhadas de chuva?**

O objetivo dessa pergunta é identificar quantas interrupções ocorreram em dias com e sem chuva, verificando se há mais interrupções quando chove. A consulta compara a participação de cada grupo (com chuva, sem chuva) no total de dias e de perturbações, a média de perturbações por dia e a proporção de dias com pelo menos uma perturbação.

In [ ]:
# A partir de quantos mm em média um dia é considerado chuvoso
limiar_mm = 1.0
coluna_chuva = "chuva_media"

# Visão com a chuva em mm e um indicador de dia chuvoso
duckdb.sql(f"""
    CREATE OR REPLACE VIEW dias AS
    SELECT *,
            {coluna_chuva} AS chuva,
            {coluna_chuva} >= {limiar_mm} AS dia_chuvoso
    FROM read_parquet('{processed_file}')
""")

duckdb.sql("""
    SELECT COUNT(*) AS dias,
            COUNT(*) FILTER (WHERE dia_chuvoso) AS dias_chuvosos,
            COUNT(*) FILTER (WHERE NOT dia_chuvoso) AS dias_sem_chuva,
    FROM dias
""").df()

Considerando como dia chuvoso aquele em que a chuva média entre as estações é de pelo menos 1 mm, houve mais dias chuvosos (1097) do que dias sem chuva (729).

In [ ]:
duckdb.sql("""
   SELECT dia_chuvoso,
            COUNT(*) AS qtd_dias,
            SUM(n_perturbacoes) AS perturbacoes,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_dias,
            ROUND(100.0 * SUM(n_perturbacoes) / SUM(SUM(n_perturbacoes)) OVER (), 1) AS pct_perturbacoes,
            ROUND(AVG(n_perturbacoes), 3) AS perturbacoes_dia,
            ROUND(100.0 * COUNT(*) FILTER (WHERE n_perturbacoes > 0) / COUNT(*), 1) AS pct_dias_interrupcao,
            ROUND(AVG(mw_interrompido), 1) AS mw_interrompido_dia,
    FROM dias
    GROUP BY dia_chuvoso
    ORDER BY dia_chuvoso DESC
""").df()

No total, foram registradas 305 perturbações, sendo 222 (72,8%) delas em dias chuvosos, que representam 60,1% dos dias. Em média, houve cerca de 0,2 perturbações por dia em dias chuvosos e 0,11 em dias sem chuva. Entre os dias sem chuva, 10% deles tiveram pelo menos uma interrupção, enquanto em dias chuvosos 17,2% deles tiveram pelo menos uma interrupção, cerca de 1,7 a 1,8 vezes o valor observado em dias sem chuva.

Esses resultados podem indicar que interrupções no subsistema sul são mais frequentes quando há chuva. Entretanto, a carga interrompida média por dia tanto em dias chuvosos quanto em dias sem chuva são semelhantes: 8,5 MW em dias chuvosos e 7,9 MW em dias sem chuva. A presença de chuva pode estar associada a mais perturbações, mas não a uma carga interrompida significativamente maior por dia.

**3. Em quais meses houveram mais interruções históricamente?**

Esta pergunta busca identificar se as interrupções têm um padrão sazonal, que é o ponto de partida para a segunda hipótese do trabalho: que os períodos do ano com mais chuva e tempestades também concentram mais interrupções. Se alguns meses se destacarem, isso indica que há uma dependência do calendário que o modelo preditivo deve incorporar, e que vale comparar esses meses com o clima registrado pelo INMET. Como o período tem apenas cinco anos, um mês pode aparecer no topo por causa de um único ano com eventos extremos.

In [ ]:
duckdb.sql(f"""
    SELECT mes,
           SUM(n_perturbacoes) AS perturbacoes,
           ROUND(100.0 * SUM(n_perturbacoes) / SUM(SUM(n_perturbacoes)) OVER (), 1) AS pct_perturbacoes
    FROM read_parquet('{processed_file}')
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
    GROUP BY mes
    ORDER BY perturbacoes DESC
""").df()

Janeiro, fevereiro e novembro foram os meses com mais perturbações, 34, 33 e 32, que juntos somam cerca de um terço do total (32,5%). Julho foi o mês com menos, com 16, e abril ficou logo acima, com 19. De modo geral, o fim da primavera e o verão concentram mais interrupções que o inverno, o que é compatível com a hipótese de que a época de chuvas e tempestades pesa mais. Também é possível notar que a distribuição das perturbações não está fortemente uniforme.

**4. Meses com mais feriados têm mais interrupções?**

Esta pergunta busca verificar se o calendário de feriados ajuda a explicar as interrupções, e portanto se as variáveis de feriado integradas à base têm utilidade para o modelo. Feriados geralmente mudam o padrão de consumo de energia e podem reduzir a disponibilidade de equipes de operação e manutenção, então é plausível que meses com mais feriados tenham um comportamento diferente. 

Como os feriados não se distribuem de forma proporcional no ano, e meses com muitos feriados também coincidem com certas estações, um evento pode portanto vir da sazonalidade examinada na questão anterior, e não dos feriados em si.

In [ ]:
mensal = f"""
    SELECT DATE_TRUNC('month', data) AS mes,
           COUNT(*) AS dias,
           SUM(n_perturbacoes) AS perturbacoes,
           SUM(CASE WHEN feriado_nacional = 1 OR frac_estados_feriado > 0 THEN 1 ELSE 0 END) AS dias_com_feriado
    FROM read_parquet('{processed_file}')
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
    GROUP BY 1
"""

duckdb.sql(f"""
    SELECT dias_com_feriado,
           COUNT(*) AS meses,
           ROUND(AVG(perturbacoes), 1) AS media_perturbacoes_mes,
           ROUND(AVG(perturbacoes / dias), 2) AS media_perturbacoes_dia
    FROM ({mensal})
    GROUP BY dias_com_feriado
    ORDER BY dias_com_feriado
""").df()

Os meses não mostram relação clara entre a quantidade de feriados e o número de interrupções, com a média mensal de perturbações entre 4,2 e 6,0 em todos os grupos, e sem crescimento constante ou evidente com os feriados.

**5. Há mais interrupções em dias com chuva extrema?**

O objetivo dessa pergunta é verificar se há mais interrupções registradas em dias com chuva extrema em relação aos demais dias de chuva. Foi considerado extremo o dia em que a chuva média entre as estações está acima do percentil 90 da distribuição dos dias chuvosos, o que corresponde a 17,5 mm. O percentil foi calculado apenas entre os dias chuvosos, e não entre todos os dias do período.

In [ ]:
chuva_extrema = 0.90
percentis_chuva = [0.80, 0.85, 0.90, 0.95, 0.99]

dias_chuvosos = duckdb.sql("SELECT chuva FROM dias WHERE dia_chuvoso").df()["chuva"]
limiares = {percentil: dias_chuvosos.quantile(percentil) for percentil in percentis_chuva}
limiar_extremo = limiares[chuva_extrema]

In [ ]:
duckdb.sql(f"""
    SELECT dia_chuvoso,
            chuva >= {limiar_extremo} AS acima_limiar,
            ROUND(MIN(chuva), 2) AS chuva_min_mm,
            ROUND(MAX(chuva), 2) AS chuva_max_mm,
            COUNT(*) AS dias,
            SUM(n_perturbacoes) AS perturbacoes,
            ROUND(AVG(n_perturbacoes), 3) AS perturbacoes_dia,
            ROUND(100.0 * COUNT(*) FILTER (WHERE n_perturbacoes > 0) / COUNT(*), 1) AS pct_dias_interrupcao,
            ROUND(AVG(mw_interrompido), 1) AS mw_medio_dia,
    FROM dias
    GROUP BY dia_chuvoso, acima_limiar
    ORDER BY dia_chuvoso, acima_limiar
""").df()

Os 110 dias de chuva extrema correspondem a aproximadamente 10% dos 1097 dias chuvosos. A taxa de perturbações por dia aumenta com a presença de chuva, com 0,11 em dias sem chuva; 0,18 em dias com chuva não extrema; e 0,37 em dias com chuva extrema. A taxa nos dias extremos é cerca de 2 vezes a dos demais dias chuvosos. Do mesmo modo, a proporção de dias com pelo menos uma interrupção é maior em dias com chuva extrema (28,2%) em relação aos demais dias chuvosos (16%).

Entretanto, apenas 41 das 202 perturbações registradas em dias chuvosos ocorreram em dias de chuva extrema, cerca de 20% das pertubrações em dias chuvosos. Em relação à carga interrompida, a média foi de 16,8 MW por dia em dias com chuva extrema, 7,6 MW nos demais dias chuvosos, e 7,9 MW em dias sem chuva. Portanto, é possível que as perturbações registradas em dias com chuva extrema no subsistema sul causem estragos maiores, porém a pouca quantidade de dias com chuva extrema pode enviesar a carga interrompida média e a quantidade de dias com interrupção.

**Comparação com limiares de chuva extrema**

Também foi realizada uma comparação dos resultados com diferentes limiares para chuva extrema, usando os percentis 80, 85, 90, 95 e 99 da chuva dos dias chuvosos. Em cada caso, os dias acima do limiar foram comparados com os demais dias chuvosos.

In [ ]:
res_limiar = """
    SELECT {percentil} AS percentil,
            ROUND({limiar}, 1) AS limiar_mm,
            COUNT(*) FILTER (WHERE chuva >= {limiar}) AS dias_extremos,
            COUNT(*) FILTER (WHERE chuva < {limiar}) AS dias_demais,
            ROUND(AVG(n_perturbacoes) FILTER (WHERE chuva >= {limiar}), 3) AS perturbacoes_dia_extremos,
            ROUND(AVG(n_perturbacoes) FILTER (WHERE chuva < {limiar}), 3) AS perturbacoes_dia_demais,
            ROUND(100.0 * COUNT(*) FILTER (WHERE chuva >= {limiar} AND n_perturbacoes > 0) 
                / COUNT(*) FILTER (WHERE chuva >= {limiar}), 1) AS pct_interrupcao_extremos,
            ROUND(100.0 * COUNT(*) FILTER (WHERE chuva < {limiar} AND n_perturbacoes > 0)
                / COUNT(*) FILTER (WHERE chuva < {limiar}), 1) AS pct_interrupcao_demais
    FROM dias
    WHERE dia_chuvoso
"""

escala = pd.concat([duckdb.sql(res_limiar.format(percentil=percentil, limiar=limiar)).df() for percentil, limiar in limiares.items()], ignore_index=True)

escala

In [ ]:
pos = np.arange(len(escala))
largura = 0.38

fig, axis = plt.subplots(figsize=(8, 4))
extremos = axis.bar(pos + largura / 2, escala["perturbacoes_dia_extremos"], largura, label="Dias acima do limiar", color="#3c6fbb")
demais = axis.bar(pos - largura / 2, escala["perturbacoes_dia_demais"], largura, label="Demais dias chuvosos", color="#8896a5")
axis.bar_label(extremos, labels=[f"{n} dias" for n in escala["dias_extremos"]], fontsize=8, padding=2)
axis.bar_label(demais, labels=[f"{n} dias" for n in escala["dias_demais"]], fontsize=8, padding=2)
axis.set_xticks(pos, [f"P{round(p * 100)}\n≥ {l} mm" for p, l in zip(escala["percentil"], escala["limiar_mm"])])
axis.set_ylabel("Perturbações por dia (média)")
axis.set_title("Perturbações por dia - chuva acima de cada percentil em relação aos demais dias chuvosos", fontsize=10)
axis.margins(y=0.12)
axis.legend(frameon=False, loc="upper left")
fig.tight_layout()

No gráfico acima, os dias acima do limiar tiveram mais perturbações por dia do que os demais dias chuvosos em todos os limiares testados. A diferença de perturbações por dia entre dias com chuva extrema e os demais dias com chuva cresce do percentil 80 ao 95. No P99, a diferença é a menor entre os percentis escolhidos, indicando que a média das pertubações por dia não é enviesada pelos dias com maiores quantidades de chuva.

### Teste por estação do ano

Na pergunta 3, os meses com mais perturbações registradas foram janeiro, fevereiro e novembro. Para testar se há sazonalidade no número de perturbações, os meses foram agrupados nas quatro estações do ano.

In [ ]:
# Agrupamento por estação

estacoes = duckdb.sql(f"""
    SELECT CASE WHEN mes IN (12, 1, 2) THEN 'Verão'
                WHEN mes IN (3, 4, 5)  THEN 'Outono'
                WHEN mes IN (6, 7, 8)  THEN 'Inverno'
                ELSE 'Primavera' END AS estacao,
           COUNT(*) AS dias,
           ROUND(AVG(chuva), 2) AS chuva_media_mm,
           ROUND(100.0 * COUNT(*) FILTER (WHERE dia_chuvoso) / COUNT(*), 1) AS pct_dias_chuvosos,
           SUM(n_perturbacoes) AS perturbacoes,
           ROUND(100.0 * SUM(n_perturbacoes) / SUM(SUM(n_perturbacoes)) OVER (), 1) AS pct_perturbacoes,
           ROUND(AVG(n_perturbacoes), 3) AS perturbacoes_dia
    FROM dias
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
    GROUP BY estacao
    ORDER BY chuva_media_mm DESC
""").df()

estacoes

As estações com maior proporção de dias chuvosos (verão e primavera) concentram mais interrupções por dia, e o inverno, mais seco, tem a menor taxa, o que é compatível com a hipótese de que mais chuva se associa a mais interrupções. Porém, as estações citadas são mais chuvosas em frequência (mais dias com chuva), mas não em quantidade (mm). 

**Teste qui-quadrado por estação do ano**

Hipótese nula (H0): a taxa de perturbações por dia é a mesma nas quatro estações.

Hipótese alternativa (H1): pelo menos uma estação tem taxa diferente. 

Como as estações não têm exatamente o mesmo número de dias, o número esperado de perturbações em cada uma é proporcional aos seus dias total de perturbações. O teste tem 3 graus de liberdade (4 estações − 1). Definimos uma significância de 5%, rejeitando H0 se o p-valor for menor que 0,05.

In [ ]:
observado = estacoes["perturbacoes"].astype(float)
esperado = observado.sum() * estacoes["dias"] / estacoes["dias"].sum()

estatistica, p_valor = chisquare(observado, f_exp=esperado)

print(f"qui-quadrado = {estatistica:.2f}\ngraus de liberdade = {len(estacoes) - 1}\np-valor = {p_valor:.4f}")
pd.DataFrame({
              "estacao": estacoes["estacao"],
              "dias": estacoes["dias"],
              "observado": observado,
              "esperado": esperado.round(1)
            })

O teste qui-quadrado não demonstrou evidência de que a taxa de perturbações por dia seja diferente entre as estações do ano. Como p = 0,1388 > 0,05, não rejeitamos a hipótese nula pois as diferenças observadas são compatíveis com a variação esperada ao acaso.

As taxas de perturbação por dia parecem induzir a hipótese, mas a diferença entre elas não é estatisticamente significativa. Portanto, o padrão por estação é um indício não conclusivo. Entretanto, não rejeitar H0 também não prova que as taxas são iguais, apenas que os dados não permitem afirmar que elas diferem significativamente entre as estações do ano.